In [ ]:
#!pip install groq

In [ ]:
from dotenv import load_dotenv
import os
import re
from groq import Groq

# Load API keys from .env file
load_dotenv()

# Initialize Groq client — uses llama-3.1-8b-instant model
client = Groq(api_key=os.getenv("GROQ_API_KEY"))
print(f"Key loaded: {os.getenv('GROQ_API_KEY')[:8]}...")

# System prompt for solo recommendation mode
# Strict rules prevent the LLM from hallucinating movies outside the candidates list
SOLO_PROMPT = """You are a movie recommendation assistant.
CRITICAL RULES — follow exactly:
- Return EXACTLY 3 recommendations, numbered 1, 2, 3
- Use ONLY movies from the candidates list provided
- Do NOT add any introduction, explanation, or commentary
- Do NOT say anything before or after the 3 recommendations
- Use EXACTLY this format:

1. Title (Year)
Why: one sentence
Match: X/10

2. Title (Year)
Why: one sentence
Match: X/10

3. Title (Year)
Why: one sentence
Match: X/10"""

# System prompt for couple mode
# Explains why each person will enjoy the movie and what they connect over
COUPLE_PROMPT = """You are a couple movie matchmaker.
Always recommend exactly 3 movies from the provided list.
For each movie explain why Person A will enjoy it, why Person B will enjoy it,
what they will both connect over, and a couple match score out of 10."""

def generate(candidates, query, mode="solo"):
    """
    Generate movie recommendations using Groq LLM.
    Includes hallucination detection for both solo and couple mode.

    Args:
        candidates: list of top 5 ranked movies from embeddings step
        query: user preference string built by build_query()
        mode: 'solo' for single user, 'couple' for two users

    Returns:
        LLM response string with 3 movie recommendations and explanations
    """
    # Select correct system prompt based on mode
    prompt = SOLO_PROMPT if mode == "solo" else COUPLE_PROMPT

    # Build valid titles for hallucination detection
    valid_titles = [c["title"].lower() for c in candidates[:5]]

    # Build context string with user preferences and candidate movies
    context = f"User preferences: {query}\n\nCandidates:\n"
    for i, c in enumerate(candidates[:5], 1):  # max 5 candidates sent to LLM
        context += f"{i}. {c['title']} ({c['year']}) | Rating: {c['rating']} | "
        context += f"Genres: {', '.join(c['genres'])} | Cast: {', '.join(c['cast'])}\n"
        context += f"   Overview: {c['overview'][:200]}\n"  # truncate overview to 200 chars

    response = client.chat.completions.create(
        model="llama-3.1-8b-instant",
        max_tokens=600,
        messages=[{"role": "system", "content": prompt},
                  {"role": "user", "content": context}]
    )
    output = response.choices[0].message.content

    # Hallucination detection — works for both solo and couple mode
    recommended = re.findall(r'\d+\.\s+(.*?)\s+\(\d{4}\)', output)
    hallucinated = [t for t in recommended
                    if not any(t.lower() in v or v in t.lower() for v in valid_titles)]

    if hallucinated:
        print(f"Hallucination detected ({mode} mode): {hallucinated}")
        print(f"Valid titles: {valid_titles}")
        print(f"Total: {len(hallucinated)}/{len(recommended)} recommendations hallucinated")
    else:
        print(f"No hallucinations detected ({mode} mode) — all {len(recommended)} recommendations valid")

    return output

In [ ]:
# Test 1 — solo mode
candidates = [
    {"title": "Inception", "year": "2010", "rating": 8.8,
     "genres": ["Sci-Fi", "Thriller"], "cast": ["Leonardo DiCaprio"],
     "overview": "A thief who steals corporate secrets through dream-sharing technology."},
    {"title": "The Dark Knight", "year": "2008", "rating": 9.0,
     "genres": ["Action", "Crime"], "cast": ["Christian Bale"],
     "overview": "Batman fights the Joker in Gotham City."},
    {"title": "Interstellar", "year": "2014", "rating": 8.6,
     "genres": ["Sci-Fi", "Drama"], "cast": ["Matthew McConaughey"],
     "overview": "A team of explorers travel through a wormhole in space."},
]

query = "I love mind-bending sci-fi movies with great visuals"
result = generate(candidates, query, mode="solo")
print("Test 1 — Solo mode:")
print(result)
print()

In [ ]:
# Test 2 — couple mode
query = """Person A (60%) enjoys Sci-Fi, Action, mood: Adventurous, loved: Interstellar.
Person B (40%) enjoys Romance, Drama, mood: Romantic, loved: Notting Hill.
Find movies both would genuinely enjoy."""

result = generate(candidates, query, mode="couple")
print("Test 2 — Couple mode:")
print(result)
print()

In [ ]:
# Test 3 — verify output is not empty
result = generate(candidates, query, mode="solo")
print(f"Test 3 — Output not empty: {len(result) > 0}")
print(f"Output length: {len(result)} characters")